# Team Features

Compile all feature engineering into a model-ready dataframe. 

In [1]:
SEASON = 2025

### Previous Tournament Results

In [2]:
import pandas as pd

pd.set_option('display.max_columns', 100)

df = pd.read_csv(r'..\data\preprocessed\kaggle\tournament_results.csv')

df = df.loc[(~df['Season'].isin([2020])) & (df['Season'] < SEASON), :].reset_index(drop=True)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results
0,2012,1101,Abilene Chr,-1.0,-1.0
1,2012,1102,Air Force,-1.0,-1.0
2,2012,1103,Akron,0.0,-0.5
3,2012,1104,Alabama,-1.0,-1.0
4,2012,1105,Alabama A&M,-1.0,-1.0
...,...,...,...,...,...
4555,2024,1476,Stonehill,-1.0,-1.0
4556,2024,1477,East Texas A&M,-1.0,-1.0
4557,2024,1478,Le Moyne,-1.0,-1.0
4558,2024,1479,Mercyhurst,-1.0,-1.0


### Barttorvik Ratings

In [3]:
df_barttorvik = pd.read_csv(r'..\data\preprocessed\barttorvik\barttorvik.csv')

df_barttorvik

,Season,TEAM,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB
0,2012,Kentucky,0.941176,119.7,88.5,31.2,0.9702,53.4,41.6,17.2,18.1,38.4,40.0,25.6,66.1,11.3
1,2012,Ohio St.,0.794118,115.5,85.5,30.0,0.9695,52.5,46.3,17.4,22.5,35.7,37.0,28.6,68.1,7.7
2,2012,Kansas,0.818182,114.7,88.1,26.6,0.9542,54.0,43.9,19.6,20.7,34.9,41.1,34.3,67.9,8.2
3,2012,Michigan St.,0.794118,112.7,86.7,26.0,0.9532,52.7,43.0,19.8,19.7,37.2,39.0,34.2,66.1,8.6
4,2012,North Carolina,0.852941,115.8,89.6,26.2,0.9502,50.0,45.1,16.2,18.5,40.5,38.1,22.0,72.7,8.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4225,2024,Stonehill,0.129032,90.4,114.2,-23.8,0.0638,46.7,52.7,19.5,16.6,22.5,22.6,29.4,68.1,-22.0
4226,2024,St. Francis PA,0.266667,93.1,118.0,-24.9,0.0620,47.2,53.0,21.2,17.1,32.9,32.6,35.4,65.5,-18.6
4227,2024,IUPUI,0.187500,92.1,116.9,-24.8,0.0610,46.5,58.2,21.3,18.5,30.0,33.2,33.4,67.3,-21.6
4228,2024,Coppin St.,0.068966,85.0,111.2,-26.2,0.0437,42.1,51.3,22.9,21.8,27.0,31.1,38.3,66.4,-23.0


In [4]:
df_spellings = pd.read_csv(
    r'..\data\unprocessed\kaggle\MTeamSpellings.csv', 
    encoding='cp1252'  # fixes issue with fancy quotes
)

df_spellings.loc[df_spellings.shape[0]] = ['fdu', 1192]

df_spellings

,TeamNameSpelling,TeamID
0,a&m-corpus chris,1394
1,a&m-corpus christi,1394
2,abilene chr,1101
3,abilene christian,1101
4,abilene-christian,1101
...,...,...
1173,youngstown st.,1464
1174,youngstown state,1464
1175,youngstown-st,1464
1176,youngstown-state,1464


In [5]:
from fuzzywuzzy.fuzz import token_sort_ratio
from fuzzywuzzy import process
from tqdm.autonotebook import tqdm

team_spellings = df_spellings['TeamNameSpelling'].unique()
barttorvik_teams = df_barttorvik['TEAM'].unique()

df_match = pd.DataFrame(
    [
        [
            barttorvik_team,
            *process.extract(
                barttorvik_team,
                team_spellings,
                scorer=token_sort_ratio,
                limit=1
            )[0][:2]
        ] for barttorvik_team in tqdm(barttorvik_teams)
    ],
    columns=['Barttorvik Team', 'Team Spelling', 'Match Score']
).sort_values('Match Score', ignore_index=True)

df_match.head(25)

C:\Users\mhugh\AppData\Local\Temp\ipykernel_7984\2439321704.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


  0%|          | 0/366 [00:00<?, ?it/s]

,Barttorvik Team,Team Spelling,Match Score
0,Queens,Queens (NC),80
1,UT Rio Grande Valley,texas rio grande valley,88
2,Texas A&M Commerce,tx a&m commerce,91
3,Cal St. Bakersfield,cal state bakersfield,92
4,Mississippi Valley St.,mississippi valley state,93
5,Southeast Missouri St.,southeast missouri state,93
6,Texas A&M Corpus Chris,texas a&m-corpus christi,96
7,Kentucky,kentucky,100
8,McNeese St.,mcneese st,100
9,Morgan St.,morgan st,100


In [6]:
barttorvik_to_spelling = dict(zip(df_match['Barttorvik Team'], df_match['Team Spelling']))

len(barttorvik_to_spelling)

366

In [7]:
spelling_to_id = dict(zip(df_spellings['TeamNameSpelling'], df_spellings['TeamID']))

len(spelling_to_id)

1178

In [8]:
df_barttorvik.insert(1, 'TeamID', df_barttorvik['TEAM'].map(barttorvik_to_spelling).map(spelling_to_id))

df_barttorvik

,Season,TeamID,TEAM,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB
0,2012,1246,Kentucky,0.941176,119.7,88.5,31.2,0.9702,53.4,41.6,17.2,18.1,38.4,40.0,25.6,66.1,11.3
1,2012,1326,Ohio St.,0.794118,115.5,85.5,30.0,0.9695,52.5,46.3,17.4,22.5,35.7,37.0,28.6,68.1,7.7
2,2012,1242,Kansas,0.818182,114.7,88.1,26.6,0.9542,54.0,43.9,19.6,20.7,34.9,41.1,34.3,67.9,8.2
3,2012,1277,Michigan St.,0.794118,112.7,86.7,26.0,0.9532,52.7,43.0,19.8,19.7,37.2,39.0,34.2,66.1,8.6
4,2012,1314,North Carolina,0.852941,115.8,89.6,26.2,0.9502,50.0,45.1,16.2,18.5,40.5,38.1,22.0,72.7,8.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4225,2024,1476,Stonehill,0.129032,90.4,114.2,-23.8,0.0638,46.7,52.7,19.5,16.6,22.5,22.6,29.4,68.1,-22.0
4226,2024,1384,St. Francis PA,0.266667,93.1,118.0,-24.9,0.0620,47.2,53.0,21.2,17.1,32.9,32.6,35.4,65.5,-18.6
4227,2024,1237,IUPUI,0.187500,92.1,116.9,-24.8,0.0610,46.5,58.2,21.3,18.5,30.0,33.2,33.4,67.3,-21.6
4228,2024,1164,Coppin St.,0.068966,85.0,111.2,-26.2,0.0437,42.1,51.3,22.9,21.8,27.0,31.1,38.3,66.4,-23.0


In [9]:
df_barttorvik.loc[df_barttorvik['TeamID'].isna(), :]

,Season,TeamID,TEAM,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB


In [10]:
df = pd.merge(
    df,
    # df_barttorvik[[
    #     'Season', 
    #     'TeamID',
    #     'WIN%',
    #     'ADJOE',
    #     'ADJDE',
    #     'BARTHAG',
    #     'ADJ T.',
    # ]],
    df_barttorvik.drop(columns=['TEAM']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,-1.0,-1.0,0.407407,98.5,100.0,-1.5,0.4564,51.1,48.4,20.6,21.5,19.8,39.4,38.2,62.3,-7.6
2,2012,1103,Akron,0.0,-0.5,0.636364,105.1,96.9,8.2,0.7161,51.5,46.4,21.0,20.6,34.8,40.0,34.0,67.8,-1.7
3,2012,1104,Alabama,-1.0,-1.0,0.656250,105.0,88.1,16.9,0.8829,49.0,43.4,20.4,21.4,33.9,36.5,38.6,63.1,1.6
4,2012,1105,Alabama A&M,-1.0,-1.0,0.192308,88.4,111.1,-22.7,0.0677,45.1,49.2,23.9,20.6,30.4,35.5,52.9,67.9,-17.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4555,2024,1476,Stonehill,-1.0,-1.0,0.129032,90.4,114.2,-23.8,0.0638,46.7,52.7,19.5,16.6,22.5,22.6,29.4,68.1,-22.0
4556,2024,1477,East Texas A&M,-1.0,-1.0,0.393939,94.3,111.5,-17.2,0.1262,46.0,52.4,16.7,18.3,24.3,30.8,39.2,66.2,-14.0
4557,2024,1478,Le Moyne,-1.0,-1.0,0.468750,98.7,110.3,-11.6,0.2168,50.1,50.6,16.6,17.8,23.1,25.3,27.8,67.4,-13.8
4558,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
df.loc[df['WIN%'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2012,1109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2012,1118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2012,1121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2012,1128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4511,2024,1432,Utica,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4524,2024,1445,W Salem St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4525,2024,1446,W Texas A&M,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4558,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Barttorvik Previous Seasons

In [12]:
df_barttorvik_prev = pd.read_csv(r'..\data\preprocessed\barttorvik_full_season\barttorvik_full_season.csv')

df_barttorvik_prev

,Season,TEAM,Past Year BARTHAG,Past 4 Years BARTHAG
0,2012,Abilene Christian,NaN,NaN
1,2012,Air Force,0.5782,0.462825
2,2012,Akron,0.6049,0.677650
3,2012,Alabama,0.8419,0.780925
4,2012,Alabama A&M,0.1283,0.103600
...,...,...,...,...
5133,2025,Wright St.,0.5530,0.551775
5134,2025,Wyoming,0.5300,0.590825
5135,2025,Xavier,0.7997,0.825800
5136,2025,Yale,0.7227,0.673600


In [13]:
team_spellings = df_spellings['TeamNameSpelling'].unique()
barttorvik_prev_teams = df_barttorvik_prev['TEAM'].unique()

df_match = pd.DataFrame(
    [
        [
            barttorvik_prev_team,
            *process.extract(
                barttorvik_prev_team,
                team_spellings,
                scorer=token_sort_ratio,
                limit=1
            )[0][:2]
        ] for barttorvik_prev_team in tqdm(barttorvik_prev_teams)
    ],
    columns=['Barttorvik Prev Team', 'Team Spelling', 'Match Score']
).sort_values('Match Score', ignore_index=True)

df_match.head(25)

  0%|          | 0/367 [00:00<?, ?it/s]

,Barttorvik Prev Team,Team Spelling,Match Score
0,Queens,Queens (NC),80
1,UT Rio Grande Valley,texas rio grande valley,88
2,Saint Francis,saint francis (ny),90
3,Texas A&M Commerce,tx a&m commerce,91
4,Winston Salem St.,winston-salem-state,91
5,Cal St. Bakersfield,cal state bakersfield,92
6,Southeast Missouri St.,southeast missouri state,93
7,Mississippi Valley St.,mississippi valley state,93
8,Texas A&M Corpus Chris,texas a&m-corpus christi,96
9,Rice,rice,100


In [14]:
barttorvik_prev_to_spelling = dict(zip(df_match['Barttorvik Prev Team'], df_match['Team Spelling']))

df_barttorvik_prev.insert(1, 'TeamID', df_barttorvik_prev['TEAM'].map(barttorvik_prev_to_spelling).map(spelling_to_id))

df_barttorvik_prev

,Season,TeamID,TEAM,Past Year BARTHAG,Past 4 Years BARTHAG
0,2012,1101,Abilene Christian,NaN,NaN
1,2012,1102,Air Force,0.5782,0.462825
2,2012,1103,Akron,0.6049,0.677650
3,2012,1104,Alabama,0.8419,0.780925
4,2012,1105,Alabama A&M,0.1283,0.103600
...,...,...,...,...,...
5133,2025,1460,Wright St.,0.5530,0.551775
5134,2025,1461,Wyoming,0.5300,0.590825
5135,2025,1462,Xavier,0.7997,0.825800
5136,2025,1463,Yale,0.7227,0.673600


In [15]:
df = pd.merge(
    df,
    df_barttorvik_prev.drop(columns=['TEAM']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,-1.0,-1.0,0.407407,98.5,100.0,-1.5,0.4564,51.1,48.4,20.6,21.5,19.8,39.4,38.2,62.3,-7.6,0.5782,0.462825
2,2012,1103,Akron,0.0,-0.5,0.636364,105.1,96.9,8.2,0.7161,51.5,46.4,21.0,20.6,34.8,40.0,34.0,67.8,-1.7,0.6049,0.677650
3,2012,1104,Alabama,-1.0,-1.0,0.656250,105.0,88.1,16.9,0.8829,49.0,43.4,20.4,21.4,33.9,36.5,38.6,63.1,1.6,0.8419,0.780925
4,2012,1105,Alabama A&M,-1.0,-1.0,0.192308,88.4,111.1,-22.7,0.0677,45.1,49.2,23.9,20.6,30.4,35.5,52.9,67.9,-17.4,0.1283,0.103600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4567,2024,1476,Stonehill,-1.0,-1.0,0.129032,90.4,114.2,-23.8,0.0638,46.7,52.7,19.5,16.6,22.5,22.6,29.4,68.1,-22.0,0.1809,NaN
4568,2024,1477,East Texas A&M,-1.0,-1.0,0.393939,94.3,111.5,-17.2,0.1262,46.0,52.4,16.7,18.3,24.3,30.8,39.2,66.2,-14.0,0.1968,NaN
4569,2024,1478,Le Moyne,-1.0,-1.0,0.468750,98.7,110.3,-11.6,0.2168,50.1,50.6,16.6,17.8,23.1,25.3,27.8,67.4,-13.8,NaN,NaN
4570,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df.loc[df['Past Year BARTHAG'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2012,1109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2012,1118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2012,1121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2012,1128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4536,2024,1445,W Salem St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4537,2024,1446,W Texas A&M,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4569,2024,1478,Le Moyne,-1.0,-1.0,0.46875,98.7,110.3,-11.6,0.2168,50.1,50.6,16.6,17.8,23.1,25.3,27.8,67.4,-13.8,NaN,NaN
4570,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
df.loc[df['Past Year BARTHAG'].isna() & (df['Past 4 Years Tournament Results'] > -1.0), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG
3663,2022,1335,Penn,-1.0,-0.666667,0.428571,104.5,108.8,-4.3,0.3862,51.3,52.1,17.0,15.3,25.4,24.1,28.8,68.9,-8.8,NaN,0.647400
3792,2022,1463,Yale,-1.0,-0.666667,0.620690,99.8,98.4,1.4,0.5389,50.1,48.9,18.3,18.2,25.6,32.7,31.1,69.8,-4.9,NaN,0.627867


### Efficiency Margin Previous Seasons

In [18]:
df_em_prev = pd.read_csv(r'..\data\preprocessed\em_full_season\em_full_season.csv')

df_em_prev

,Season,TEAM,Past Year ADJEM,Past 4 Years ADJEM
0,2012,Abilene Christian,NaN,NaN
1,2012,Air Force,2.8,-1.375000
2,2012,Akron,3.7,6.675000
3,2012,Alabama,14.2,11.425000
4,2012,Alabama A&M,-16.1,-18.400000
...,...,...,...,...
5133,2025,Wright St.,2.1,2.075000
5134,2025,Wyoming,1.1,3.550000
5135,2025,Xavier,12.7,14.500000
5136,2025,Yale,8.8,6.733333


In [19]:
def match_names(team_spellings, new_data_teams):
    df_match = pd.DataFrame(
        [
            [
                new_data_team,
                *process.extract(
                    new_data_team,
                    team_spellings,
                    scorer=token_sort_ratio,
                    limit=1
                )[0][:2]
            ] for new_data_team in tqdm(new_data_teams)
        ],
        columns=['New Data Team', 'Team Spelling', 'Match Score']
    ).sort_values('Match Score', ignore_index=True)

    team_to_spelling = dict(zip(df_match['New Data Team'], df_match['Team Spelling']))

    return df_match, team_to_spelling

In [20]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_em_prev['TEAM'].unique())

df_match.head(25)

  0%|          | 0/367 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Queens,Queens (NC),80
1,UT Rio Grande Valley,texas rio grande valley,88
2,Saint Francis,saint francis (ny),90
3,Texas A&M Commerce,tx a&m commerce,91
4,Winston Salem St.,winston-salem-state,91
5,Cal St. Bakersfield,cal state bakersfield,92
6,Southeast Missouri St.,southeast missouri state,93
7,Mississippi Valley St.,mississippi valley state,93
8,Texas A&M Corpus Chris,texas a&m-corpus christi,96
9,Rice,rice,100


In [21]:
df_em_prev.insert(1, 'TeamID', df_barttorvik_prev['TEAM'].map(team_to_spelling).map(spelling_to_id))

df_em_prev

,Season,TeamID,TEAM,Past Year ADJEM,Past 4 Years ADJEM
0,2012,1101,Abilene Christian,NaN,NaN
1,2012,1102,Air Force,2.8,-1.375000
2,2012,1103,Akron,3.7,6.675000
3,2012,1104,Alabama,14.2,11.425000
4,2012,1105,Alabama A&M,-16.1,-18.400000
...,...,...,...,...,...
5133,2025,1460,Wright St.,2.1,2.075000
5134,2025,1461,Wyoming,1.1,3.550000
5135,2025,1462,Xavier,12.7,14.500000
5136,2025,1463,Yale,8.8,6.733333


In [22]:
df = pd.merge(
    df,
    df_em_prev.drop(columns=['TEAM']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG,Past Year ADJEM,Past 4 Years ADJEM
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,-1.0,-1.0,0.407407,98.5,100.0,-1.5,0.4564,51.1,48.4,20.6,21.5,19.8,39.4,38.2,62.3,-7.6,0.5782,0.462825,2.8,-1.375
2,2012,1103,Akron,0.0,-0.5,0.636364,105.1,96.9,8.2,0.7161,51.5,46.4,21.0,20.6,34.8,40.0,34.0,67.8,-1.7,0.6049,0.677650,3.7,6.675
3,2012,1104,Alabama,-1.0,-1.0,0.656250,105.0,88.1,16.9,0.8829,49.0,43.4,20.4,21.4,33.9,36.5,38.6,63.1,1.6,0.8419,0.780925,14.2,11.425
4,2012,1105,Alabama A&M,-1.0,-1.0,0.192308,88.4,111.1,-22.7,0.0677,45.1,49.2,23.9,20.6,30.4,35.5,52.9,67.9,-17.4,0.1283,0.103600,-16.1,-18.400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4591,2024,1476,Stonehill,-1.0,-1.0,0.129032,90.4,114.2,-23.8,0.0638,46.7,52.7,19.5,16.6,22.5,22.6,29.4,68.1,-22.0,0.1809,NaN,-13.2,NaN
4592,2024,1477,East Texas A&M,-1.0,-1.0,0.393939,94.3,111.5,-17.2,0.1262,46.0,52.4,16.7,18.3,24.3,30.8,39.2,66.2,-14.0,0.1968,NaN,-13.0,NaN
4593,2024,1478,Le Moyne,-1.0,-1.0,0.468750,98.7,110.3,-11.6,0.2168,50.1,50.6,16.6,17.8,23.1,25.3,27.8,67.4,-13.8,NaN,NaN,NaN,NaN
4594,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
df.loc[df['Past Year ADJEM'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG,Past Year ADJEM,Past 4 Years ADJEM
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2012,1109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2012,1118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2012,1121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2012,1128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4560,2024,1445,W Salem St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4561,2024,1446,W Texas A&M,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4593,2024,1478,Le Moyne,-1.0,-1.0,0.46875,98.7,110.3,-11.6,0.2168,50.1,50.6,16.6,17.8,23.1,25.3,27.8,67.4,-13.8,NaN,NaN,NaN,NaN
4594,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### My Rankings

In [24]:
df_rankings = pd.concat(
    (
        pd.read_parquet(fr'..\data\preprocessed\my_rankings\my_rankings_{season}.parquet')
        .assign(Season=season)
        for season in range(2012, SEASON) if season != 2020
    ),
    ignore_index=True
)

df_rankings.insert(0, 'Season', df_rankings.pop('Season'))

df_rankings.drop(columns=['Strength'], inplace=True)

df_rankings

,Season,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2012,Kentucky,3.565314,0.297701,1.166637,0.868936,66.950567
1,2012,North Carolina,3.219256,0.242387,1.124366,0.881978,73.342503
2,2012,Syracuse,3.027328,0.247189,1.132119,0.884929,67.141049
3,2012,Ohio State,2.922136,0.289967,1.133910,0.843942,68.205218
4,2012,Duke,2.910209,0.197090,1.147153,0.950063,68.353372
...,...,...,...,...,...,...,...
4224,2024,Virginia Military Institute,-2.050009,-0.226840,0.888982,1.115822,75.056385
4225,2024,Stonehill,-2.164153,-0.214380,0.914280,1.128660,69.240111
4226,2024,Coppin State,-2.203441,-0.250565,0.847038,1.097603,67.665417
4227,2024,Mississippi Valley State,-2.690286,-0.330596,0.847257,1.177853,65.666144


In [25]:
my_teams = df_rankings['Team'].unique()

df_match = pd.DataFrame(
    [
        [
            my_team,
            *process.extract(
                my_team,
                team_spellings,
                scorer=token_sort_ratio,
                limit=1
            )[0][:2]
        ] for my_team in tqdm(my_teams)
    ],
    columns=['My Team', 'Team Spelling', 'Match Score']
).sort_values('Match Score', ignore_index=True)

df_match.head(25)

  0%|          | 0/365 [00:00<?, ?it/s]

,My Team,Team Spelling,Match Score
0,Hartford Hawks,hartford,73
1,St. Francis (NY) Terriers,st francis (ny),74
2,Savannah State Tigers,savannah state,80
3,Texas A&M-Commerce,tx a&m commerce,91
4,Kentucky,kentucky,100
5,Eastern Kentucky,eastern kentucky,100
6,North Dakota,north dakota,100
7,Delaware State,delaware state,100
8,Marist,marist,100
9,Stetson,stetson,100


In [26]:
ranking_to_spelling = dict(zip(df_match['My Team'], df_match['Team Spelling']))

len(ranking_to_spelling)

365

In [27]:
df_rankings.insert(1, 'TeamID', df_rankings['Team'].map(ranking_to_spelling).map(spelling_to_id))

df_rankings

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2012,1246,Kentucky,3.565314,0.297701,1.166637,0.868936,66.950567
1,2012,1314,North Carolina,3.219256,0.242387,1.124366,0.881978,73.342503
2,2012,1393,Syracuse,3.027328,0.247189,1.132119,0.884929,67.141049
3,2012,1326,Ohio State,2.922136,0.289967,1.133910,0.843942,68.205218
4,2012,1181,Duke,2.910209,0.197090,1.147153,0.950063,68.353372
...,...,...,...,...,...,...,...,...
4224,2024,1440,Virginia Military Institute,-2.050009,-0.226840,0.888982,1.115822,75.056385
4225,2024,1476,Stonehill,-2.164153,-0.214380,0.914280,1.128660,69.240111
4226,2024,1164,Coppin State,-2.203441,-0.250565,0.847038,1.097603,67.665417
4227,2024,1290,Mississippi Valley State,-2.690286,-0.330596,0.847257,1.177853,65.666144


In [28]:
df_rankings.loc[df_rankings['TeamID'].isna(), :]

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo


In [29]:
df = pd.merge(
    df,
    df_rankings.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG,Past Year ADJEM,Past 4 Years ADJEM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,-1.0,-1.0,0.407407,98.5,100.0,-1.5,0.4564,51.1,48.4,20.6,21.5,19.8,39.4,38.2,62.3,-7.6,0.5782,0.462825,2.8,-1.375,0.155360,-0.020097,0.979952,1.000049,62.836286
2,2012,1103,Akron,0.0,-0.5,0.636364,105.1,96.9,8.2,0.7161,51.5,46.4,21.0,20.6,34.8,40.0,34.0,67.8,-1.7,0.6049,0.677650,3.7,6.675,1.338710,0.095748,1.045688,0.949940,68.483190
3,2012,1104,Alabama,-1.0,-1.0,0.656250,105.0,88.1,16.9,0.8829,49.0,43.4,20.4,21.4,33.9,36.5,38.6,63.1,1.6,0.8419,0.780925,14.2,11.425,1.574844,0.154447,1.034203,0.879756,63.968330
4,2012,1105,Alabama A&M,-1.0,-1.0,0.192308,88.4,111.1,-22.7,0.0677,45.1,49.2,23.9,20.6,30.4,35.5,52.9,67.9,-17.4,0.1283,0.103600,-16.1,-18.400,-3.145351,-0.191359,0.890988,1.082347,68.157167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4591,2024,1476,Stonehill,-1.0,-1.0,0.129032,90.4,114.2,-23.8,0.0638,46.7,52.7,19.5,16.6,22.5,22.6,29.4,68.1,-22.0,0.1809,NaN,-13.2,NaN,-2.164153,-0.214380,0.914280,1.128660,69.240111
4592,2024,1477,East Texas A&M,-1.0,-1.0,0.393939,94.3,111.5,-17.2,0.1262,46.0,52.4,16.7,18.3,24.3,30.8,39.2,66.2,-14.0,0.1968,NaN,-13.0,NaN,-1.196470,-0.161715,0.935543,1.097258,67.458552
4593,2024,1478,Le Moyne,-1.0,-1.0,0.468750,98.7,110.3,-11.6,0.2168,50.1,50.6,16.6,17.8,23.1,25.3,27.8,67.4,-13.8,NaN,NaN,NaN,NaN,-1.228214,-0.091264,1.001228,1.092492,68.446622
4594,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
df.loc[df['Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG,Past Year ADJEM,Past 4 Years ADJEM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2012,1109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2012,1118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2012,1121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2012,1128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4547,2024,1432,Utica,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4560,2024,1445,W Salem St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4561,2024,1446,W Texas A&M,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4594,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
df.loc[(df['Rating'].isna()) & (df['Past 4 Years Tournament Results'] > -1.0), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG,Past Year ADJEM,Past 4 Years ADJEM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
3298,2021,1335,Penn,NaN,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.6003,0.613750,3.6,4.150,NaN,NaN,NaN,NaN,NaN
3306,2021,1343,Princeton,NaN,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5723,0.581775,2.6,3.325,NaN,NaN,NaN,NaN,NaN
3429,2021,1463,Yale,NaN,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.7549,0.599650,10.0,3.975,NaN,NaN,NaN,NaN,NaN
4328,2024,1216,Hartford,-1.0,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0418,0.264125,-28.0,-11.575,NaN,NaN,NaN,NaN,NaN


### Starters

In [32]:
df_starters = pd.concat(
    (
        pd.read_parquet(fr'..\data\preprocessed\starters\starters_{season}.parquet')
        .assign(Season=season)
        for season in range(2012, SEASON) if season != 2020
    ),
    ignore_index=True
)

df_starters.insert(0, 'Season', df_starters.pop('Season'))

df_starters.rename(columns={'Rating': 'Starters'}, inplace=True)

df_starters

,Season,Team,Starters
0,2012,Murray State,0.583076
1,2012,Michigan State,0.561808
2,2012,Duke,0.537206
3,2012,Wichita State,0.536827
4,2012,Kentucky,0.534842
...,...,...,...
4224,2024,Stonehill,-0.400817
4225,2024,Virginia Military Institute,-0.406939
4226,2024,Buffalo,-0.422366
4227,2024,Mississippi Valley State,-0.442076


In [33]:
starters_teams = df_starters['Team'].unique()

df_match = pd.DataFrame(
    [
        [
            starters_team,
            *process.extract(
                starters_team,
                team_spellings,
                scorer=token_sort_ratio,
                limit=1
            )[0][:2]
        ] for starters_team in tqdm(starters_teams)
    ],
    columns=['Starters Team', 'Team Spelling', 'Match Score']
).sort_values('Match Score', ignore_index=True)

df_match.head(25)

  0%|          | 0/365 [00:00<?, ?it/s]

,Starters Team,Team Spelling,Match Score
0,Texas A&M-Commerce,tx a&m commerce,91
1,Murray State,murray state,100
2,Lipscomb,lipscomb,100
3,Louisiana,louisiana,100
4,Texas State,texas state,100
5,UC Irvine,uc irvine,100
6,Southern,southern,100
7,Appalachian State,appalachian state,100
8,Southeastern Louisiana,southeastern louisiana,100
9,Morgan State,morgan state,100


In [34]:
starters_to_spelling = dict(zip(df_match['Starters Team'], df_match['Team Spelling']))

len(starters_to_spelling)

365

In [35]:
df_starters.insert(1, 'TeamID', df_starters['Team'].map(starters_to_spelling).map(spelling_to_id))

df_starters

,Season,TeamID,Team,Starters
0,2012,1293,Murray State,0.583076
1,2012,1277,Michigan State,0.561808
2,2012,1181,Duke,0.537206
3,2012,1455,Wichita State,0.536827
4,2012,1246,Kentucky,0.534842
...,...,...,...,...
4224,2024,1476,Stonehill,-0.400817
4225,2024,1440,Virginia Military Institute,-0.406939
4226,2024,1138,Buffalo,-0.422366
4227,2024,1290,Mississippi Valley State,-0.442076


In [36]:
df_starters.loc[df_starters['TeamID'].isna(), :]

,Season,TeamID,Team,Starters


In [37]:
df = pd.merge(
    df,
    df_starters.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG,Past Year ADJEM,Past 4 Years ADJEM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,-1.0,-1.0,0.407407,98.5,100.0,-1.5,0.4564,51.1,48.4,20.6,21.5,19.8,39.4,38.2,62.3,-7.6,0.5782,0.462825,2.8,-1.375,0.155360,-0.020097,0.979952,1.000049,62.836286,-0.066193
2,2012,1103,Akron,0.0,-0.5,0.636364,105.1,96.9,8.2,0.7161,51.5,46.4,21.0,20.6,34.8,40.0,34.0,67.8,-1.7,0.6049,0.677650,3.7,6.675,1.338710,0.095748,1.045688,0.949940,68.483190,0.359659
3,2012,1104,Alabama,-1.0,-1.0,0.656250,105.0,88.1,16.9,0.8829,49.0,43.4,20.4,21.4,33.9,36.5,38.6,63.1,1.6,0.8419,0.780925,14.2,11.425,1.574844,0.154447,1.034203,0.879756,63.968330,0.226809
4,2012,1105,Alabama A&M,-1.0,-1.0,0.192308,88.4,111.1,-22.7,0.0677,45.1,49.2,23.9,20.6,30.4,35.5,52.9,67.9,-17.4,0.1283,0.103600,-16.1,-18.400,-3.145351,-0.191359,0.890988,1.082347,68.157167,-0.315889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4591,2024,1476,Stonehill,-1.0,-1.0,0.129032,90.4,114.2,-23.8,0.0638,46.7,52.7,19.5,16.6,22.5,22.6,29.4,68.1,-22.0,0.1809,NaN,-13.2,NaN,-2.164153,-0.214380,0.914280,1.128660,69.240111,-0.400817
4592,2024,1477,East Texas A&M,-1.0,-1.0,0.393939,94.3,111.5,-17.2,0.1262,46.0,52.4,16.7,18.3,24.3,30.8,39.2,66.2,-14.0,0.1968,NaN,-13.0,NaN,-1.196470,-0.161715,0.935543,1.097258,67.458552,-0.111816
4593,2024,1478,Le Moyne,-1.0,-1.0,0.468750,98.7,110.3,-11.6,0.2168,50.1,50.6,16.6,17.8,23.1,25.3,27.8,67.4,-13.8,NaN,NaN,NaN,NaN,-1.228214,-0.091264,1.001228,1.092492,68.446622,-0.070744
4594,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
df.loc[df['Starters'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG,Past Year ADJEM,Past 4 Years ADJEM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2012,1109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2012,1118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2012,1121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2012,1128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4547,2024,1432,Utica,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4560,2024,1445,W Salem St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4561,2024,1446,W Texas A&M,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4594,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
df.loc[(df['Starters'].isna()) & (df['Past 4 Years Tournament Results'] > -1.0), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG,Past Year ADJEM,Past 4 Years ADJEM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
3298,2021,1335,Penn,NaN,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.6003,0.613750,3.6,4.150,NaN,NaN,NaN,NaN,NaN,NaN
3306,2021,1343,Princeton,NaN,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5723,0.581775,2.6,3.325,NaN,NaN,NaN,NaN,NaN,NaN
3429,2021,1463,Yale,NaN,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.7549,0.599650,10.0,3.975,NaN,NaN,NaN,NaN,NaN,NaN
4328,2024,1216,Hartford,-1.0,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0418,0.264125,-28.0,-11.575,NaN,NaN,NaN,NaN,NaN,NaN


### Openskill Ratings

In [40]:
df_os = pd.concat(
    (
        pd.read_parquet(fr'..\data\preprocessed\os_rankings\os_rankings_{season}.parquet')
        .assign(Season=season)
        for season in range(2012, SEASON) if season != 2020
    ),
    ignore_index=True
)

df_os.insert(0, 'Season', df_os.pop('Season'))

df_os.drop(columns=['Sigma'], inplace=True)

df_os

,Season,Team,Mu,OS Rating
0,2012,Kentucky,56.478949,44.081022
1,2012,Michigan State,53.318881,42.492050
2,2012,Syracuse,55.055612,42.477346
3,2012,Missouri,53.231065,41.582217
4,2012,North Carolina,51.469704,40.094136
...,...,...,...,...
4224,2024,Houston Christian,-1.396884,-14.103055
4225,2024,IUPUI,-1.936764,-14.554820
4226,2024,Detroit Mercy,-1.438196,-14.876011
4227,2024,Coppin State,-2.556670,-15.339692


In [41]:
os_teams = df_os['Team'].unique()

df_match = pd.DataFrame(
    [
        [
            os_team,
            *process.extract(
                os_team,
                team_spellings,
                scorer=token_sort_ratio,
                limit=1
            )[0][:2]
        ] for os_team in tqdm(os_teams)
    ],
    columns=['OS Team', 'Team Spelling', 'Match Score']
).sort_values('Match Score', ignore_index=True)

df_match.head(25)

  0%|          | 0/365 [00:00<?, ?it/s]

,OS Team,Team Spelling,Match Score
0,Hartford Hawks,hartford,73
1,St. Francis (NY) Terriers,st francis (ny),74
2,Savannah State Tigers,savannah state,80
3,Texas A&M-Commerce,tx a&m commerce,91
4,Kentucky,kentucky,100
5,Presbyterian,presbyterian,100
6,Boston College,boston college,100
7,Columbia,columbia,100
8,Ball State,ball state,100
9,Eastern Michigan,eastern michigan,100


In [42]:
os_to_spelling = dict(zip(df_match['OS Team'], df_match['Team Spelling']))

len(os_to_spelling)

365

In [43]:
df_os.insert(1, 'TeamID', df_os['Team'].map(os_to_spelling).map(spelling_to_id))

df_os

,Season,TeamID,Team,Mu,OS Rating
0,2012,1246,Kentucky,56.478949,44.081022
1,2012,1277,Michigan State,53.318881,42.492050
2,2012,1393,Syracuse,55.055612,42.477346
3,2012,1281,Missouri,53.231065,41.582217
4,2012,1314,North Carolina,51.469704,40.094136
...,...,...,...,...,...
4224,2024,1223,Houston Christian,-1.396884,-14.103055
4225,2024,1237,IUPUI,-1.936764,-14.554820
4226,2024,1178,Detroit Mercy,-1.438196,-14.876011
4227,2024,1164,Coppin State,-2.556670,-15.339692


In [44]:
df = pd.merge(
    df,
    df_os.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG,Past Year ADJEM,Past 4 Years ADJEM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,-1.0,-1.0,0.407407,98.5,100.0,-1.5,0.4564,51.1,48.4,20.6,21.5,19.8,39.4,38.2,62.3,-7.6,0.5782,0.462825,2.8,-1.375,0.155360,-0.020097,0.979952,1.000049,62.836286,-0.066193,24.127044,11.638416
2,2012,1103,Akron,0.0,-0.5,0.636364,105.1,96.9,8.2,0.7161,51.5,46.4,21.0,20.6,34.8,40.0,34.0,67.8,-1.7,0.6049,0.677650,3.7,6.675,1.338710,0.095748,1.045688,0.949940,68.483190,0.359659,35.006792,23.723917
3,2012,1104,Alabama,-1.0,-1.0,0.656250,105.0,88.1,16.9,0.8829,49.0,43.4,20.4,21.4,33.9,36.5,38.6,63.1,1.6,0.8419,0.780925,14.2,11.425,1.574844,0.154447,1.034203,0.879756,63.968330,0.226809,39.809098,29.069662
4,2012,1105,Alabama A&M,-1.0,-1.0,0.192308,88.4,111.1,-22.7,0.0677,45.1,49.2,23.9,20.6,30.4,35.5,52.9,67.9,-17.4,0.1283,0.103600,-16.1,-18.400,-3.145351,-0.191359,0.890988,1.082347,68.157167,-0.315889,0.507554,-11.288135
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4591,2024,1476,Stonehill,-1.0,-1.0,0.129032,90.4,114.2,-23.8,0.0638,46.7,52.7,19.5,16.6,22.5,22.6,29.4,68.1,-22.0,0.1809,NaN,-13.2,NaN,-2.164153,-0.214380,0.914280,1.128660,69.240111,-0.400817,-0.691537,-12.746634
4592,2024,1477,East Texas A&M,-1.0,-1.0,0.393939,94.3,111.5,-17.2,0.1262,46.0,52.4,16.7,18.3,24.3,30.8,39.2,66.2,-14.0,0.1968,NaN,-13.0,NaN,-1.196470,-0.161715,0.935543,1.097258,67.458552,-0.111816,9.838817,-1.497539
4593,2024,1478,Le Moyne,-1.0,-1.0,0.468750,98.7,110.3,-11.6,0.2168,50.1,50.6,16.6,17.8,23.1,25.3,27.8,67.4,-13.8,NaN,NaN,NaN,NaN,-1.228214,-0.091264,1.001228,1.092492,68.446622,-0.070744,14.058704,3.316992
4594,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
df.loc[df['OS Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG,Past Year ADJEM,Past 4 Years ADJEM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2012,1109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2012,1118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2012,1121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2012,1128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4547,2024,1432,Utica,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4560,2024,1445,W Salem St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4561,2024,1446,W Texas A&M,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4594,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Map to Matchups

In [46]:
df_mod = pd.read_csv(r'..\data\unprocessed\kaggle\MNCAATourneyDetailedResults.csv')[['Season', 'DayNum', 'WTeamID', 'LTeamID']]

df_mod = df_mod.loc[df_mod['Season'].between(2012, SEASON, inclusive='left'), :].reset_index(drop=True)

# fix 2021 dates
df_mod.loc[
    (df_mod['Season'] == 2021) & 
    (df_mod['DayNum'] < 140), 
    'DayNum'
] = df_mod.loc[
    (df_mod['Season'] == 2021) & 
    (df_mod['DayNum'] < 140), 
    'DayNum'
] - 1

df_mod.loc[
    (df_mod['Season'] == 2021) & 
    (df_mod['DayNum'].between(140, 150)), 
    'DayNum'
] = df_mod.loc[
    (df_mod['Season'] == 2021) & 
    (df_mod['DayNum'].between(140, 150)), 
    'DayNum'
] - 2

# get rid of play-in games
df_mod = df_mod.loc[df_mod['DayNum'] > 135, :].reset_index(drop=True)

df_mod

,Season,DayNum,WTeamID,LTeamID
0,2012,136,1124,1355
1,2012,136,1160,1424
2,2012,136,1211,1452
3,2012,136,1231,1308
4,2012,136,1235,1163
...,...,...,...,...
750,2024,146,1301,1181
751,2024,146,1345,1397
752,2024,152,1163,1104
753,2024,152,1345,1301


Get round of each game

In [47]:
df_mod['Round'] = 1
df_mod.loc[df_mod['DayNum'].between(138, 139), 'Round'] = 2
df_mod.loc[df_mod['DayNum'].between(140, 144), 'Round'] = 3
df_mod.loc[df_mod['DayNum'].between(145, 149), 'Round'] = 4
df_mod.loc[df_mod['DayNum'].between(150, 153), 'Round'] = 5
df_mod.loc[df_mod['DayNum'] == 154, 'Round'] = 6

df_mod

,Season,DayNum,WTeamID,LTeamID,Round
0,2012,136,1124,1355,1
1,2012,136,1160,1424,1
2,2012,136,1211,1452,1
3,2012,136,1231,1308,1
4,2012,136,1235,1163,1
...,...,...,...,...,...
750,2024,146,1301,1181,4
751,2024,146,1345,1397,4
752,2024,152,1163,1104,5
753,2024,152,1345,1301,5


Check if there are any irregularities of number of games in a round

In [48]:
for season in range(2012, SEASON):
    for round_ in range(1, 7):
        if df_mod.loc[(df_mod['Season'] == season) & (df_mod['Round'] == round_), :].shape[0] != 2**(6 - round_):
            print(f"{season}, {round_} : {df_mod.loc[(df_mod['Season'] == season) & (df_mod['Round'] == round_), :].shape[0]}")

2020, 1 : 0
2020, 2 : 0
2020, 3 : 0
2020, 4 : 0
2020, 5 : 0
2020, 6 : 0
2021, 1 : 31


Remap to Team A / Team B format

In [49]:
df_mod = pd.DataFrame({
    'Season': list(df_mod['Season'])*2,
    'Round': list(df_mod['Round'])*2,
    'Result': [1 for _ in range(df_mod.shape[0])] + [-1 for _ in range(df_mod.shape[0])],
    'Team A ID': list(df_mod['WTeamID']) + list(df_mod['LTeamID']),
    'Team B ID': list(df_mod['LTeamID']) + list(df_mod['WTeamID']),
})

df_mod

,Season,Round,Result,Team A ID,Team B ID
0,2012,1,1,1124,1355
1,2012,1,1,1160,1424
2,2012,1,1,1211,1452
3,2012,1,1,1231,1308
4,2012,1,1,1235,1163
...,...,...,...,...,...
1505,2024,4,-1,1181,1301
1506,2024,4,-1,1397,1345
1507,2024,5,-1,1104,1163
1508,2024,5,-1,1301,1345


Get Head-to-Head (Omitted)

In [50]:
# df_h2h = pd.read_csv('../data/preprocessed/men_h2h/men_h2h.csv')

# df_h2h

In [51]:
# h2h_teams = df_h2h['Team A'].unique()

# df_match = pd.DataFrame(
#     [
#         [
#             h2h_team,
#             *process.extract(
#                 h2h_team,
#                 team_spellings,
#                 scorer=token_sort_ratio,
#                 limit=1
#             )[0][:2]
#         ] for h2h_team in tqdm(h2h_teams)
#     ],
#     columns=['Head to Head Team', 'Team Spelling', 'Match Score']
# ).sort_values('Match Score', ignore_index=True)

# df_match.head(25)

In [52]:
# h2h_to_spelling = dict(zip(df_match['Head to Head Team'], df_match['Team Spelling']))

# df_h2h.insert(df_h2h.columns.get_loc('Team A'), 'Team A ID', df_h2h['Team A'].map(h2h_to_spelling).map(spelling_to_id))

# df_h2h.insert(df_h2h.columns.get_loc('Team B'), 'Team B ID', df_h2h['Team B'].map(h2h_to_spelling).map(spelling_to_id))

# df_h2h

In [53]:
# df_mod = pd.merge(
#     df_mod,
#     df_h2h[['Season', 'Team A ID', 'Team B ID', 'Head to Head', 'Common Opps']],
#     how='left',
#     on=['Season', 'Team A ID', 'Team B ID'],
# )

# df_mod

Get team names

In [54]:
df_teams = pd.read_csv(r'..\data\unprocessed\kaggle\MTeams.csv')

df_teams

,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2025
1,1102,Air Force,1985,2025
2,1103,Akron,1985,2025
3,1104,Alabama,1985,2025
4,1105,Alabama A&M,2000,2025
...,...,...,...,...
375,1476,Stonehill,2023,2025
376,1477,East Texas A&M,2023,2025
377,1478,Le Moyne,2024,2025
378,1479,Mercyhurst,2025,2025


In [55]:
id_to_team = dict(zip(df_teams['TeamID'], df_teams['TeamName']))

df_mod.insert(df_mod.columns.get_loc('Team A ID') + 1, 'Team A', df_mod['Team A ID'].map(id_to_team))
df_mod.insert(df_mod.columns.get_loc('Team B ID') + 1, 'Team B', df_mod['Team B ID'].map(id_to_team))

df_mod

,Season,Round,Result,Team A ID,Team A,Team B ID,Team B
0,2012,1,1,1124,Baylor,1355,S Dakota St
1,2012,1,1,1160,Colorado,1424,UNLV
2,2012,1,1,1211,Gonzaga,1452,West Virginia
3,2012,1,1,1231,Indiana,1308,New Mexico St
4,2012,1,1,1235,Iowa St,1163,Connecticut
...,...,...,...,...,...,...,...
1505,2024,4,-1,1181,Duke,1301,NC State
1506,2024,4,-1,1397,Tennessee,1345,Purdue
1507,2024,5,-1,1104,Alabama,1163,Connecticut
1508,2024,5,-1,1301,NC State,1345,Purdue


Map features

In [56]:
team_a_features = pd.merge(
    df_mod[['Season', 'Team A ID']],
    df.drop(columns=['Team']),
    how='left',
    left_on=['Season', 'Team A ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team A ID', 'TeamID'])

team_b_features = pd.merge(
    df_mod[['Season', 'Team B ID']],
    df.drop(columns=['Team']),
    how='left',
    left_on=['Season', 'Team B ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team B ID', 'TeamID'])

df_features = team_a_features - team_b_features

df_features['Team A ADJOE Team B ADJDE'] = team_a_features['ADJOE'] + team_b_features['ADJDE']
df_features['Team B ADJOE Team A ADJDE'] = team_b_features['ADJOE'] + team_a_features['ADJDE']

df_features['Team A Offense Team B Defense'] = team_a_features['Adjusted Offense'] + team_b_features['Adjusted Defense']
df_features['Team B Offense Team A Defense'] = team_b_features['Adjusted Offense'] + team_a_features['Adjusted Defense']

df_features['Team A BARTHAG'] = team_a_features['BARTHAG']
df_features['Team B BARTHAG'] = team_b_features['BARTHAG']

df_features

,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG,Past Year ADJEM,Past 4 Years ADJEM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Team A ADJOE Team B ADJDE,Team B ADJOE Team A ADJDE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A BARTHAG,Team B BARTHAG
0,0.0,1.250000,0.006629,2.7,-7.1,9.8,0.1420,-1.2,-2.7,5.0,2.1,7.1,-3.4,2.2,0.4,7.4,0.0703,0.428300,2.4,20.000,2.338760,0.067615,0.002898,-0.064718,0.583440,0.117964,7.605073,7.704642,215.1,205.3,2.102757,2.035141,0.9090,0.7670
1,-1.0,-1.000000,-0.093750,-5.1,1.9,-7.0,-0.1138,-3.9,0.9,-0.3,-2.3,-3.3,7.2,-0.9,-4.0,-3.1,-0.0848,-0.168775,-4.6,-8.075,-1.031018,-0.086742,-0.061467,0.025275,-3.751519,-0.226425,-7.098771,-6.444547,196.5,203.5,1.942409,2.029151,0.7615,0.8753
2,0.0,-0.750000,0.212702,-0.3,-1.2,0.9,0.0132,4.7,-2.8,0.6,-0.7,-6.4,8.9,-4.7,0.7,2.3,-0.0286,-0.026625,-3.0,-2.850,0.274774,0.029706,0.004889,-0.024816,0.670748,0.164332,6.727094,5.540592,206.1,205.2,2.033816,2.004111,0.8612,0.8480
3,0.0,0.000000,0.030303,13.0,-0.9,13.9,0.1917,4.0,-0.1,-1.6,-0.1,-5.2,-6.2,1.0,-3.3,7.6,0.1886,0.008000,7.6,1.450,1.508085,0.104455,0.097244,-0.007210,-3.018990,0.115812,7.908734,8.270663,216.5,202.6,2.116803,2.012348,0.9184,0.7267
4,-7.0,-3.250000,0.081439,0.9,1.0,-0.1,-0.0023,2.7,3.9,-0.8,0.7,-4.6,4.3,0.8,3.0,1.4,-0.1929,-0.200225,-13.4,-13.550,0.155641,0.007995,0.021385,0.013390,2.322124,-0.035981,3.232739,3.208781,205.5,205.6,2.038216,2.030221,0.8553,0.8576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1505,1.0,2.000000,0.138889,6.8,-4.4,11.2,0.1313,4.5,-1.5,0.6,-1.0,3.2,1.3,-4.3,-1.1,4.0,0.0800,0.136625,6.0,10.200,0.102572,0.121062,0.073952,-0.047109,-0.912177,0.151976,5.024188,4.103541,222.1,210.9,2.206684,2.085623,0.9262,0.7949
1506,2.0,0.333333,-0.128788,-10.6,-3.5,-7.1,-0.0265,-4.5,-2.3,-1.9,4.9,-5.0,-8.5,12.8,1.6,-5.2,0.0068,-0.016675,-0.5,-1.725,-0.279607,-0.035856,-0.077972,-0.042117,1.337846,-0.133075,-5.394708,-4.408191,210.3,217.4,2.099227,2.135083,0.9375,0.9640
1507,-4.0,-0.666667,-0.255515,-1.7,8.2,-9.9,-0.0577,-0.8,4.8,1.1,-0.6,-1.6,1.9,7.1,8.0,-7.5,-0.0095,-0.001675,-3.2,-0.525,-0.392997,-0.059663,0.001401,0.061064,7.406847,-0.265038,-11.210705,-10.392373,219.0,228.9,2.178061,2.237724,0.9115,0.9692
1508,0.0,-1.333333,-0.267677,-12.1,6.5,-18.6,-0.1691,-5.3,2.8,-2.8,3.9,-8.8,-10.3,8.9,0.3,-11.7,-0.1061,-0.129800,-9.8,-9.200,-1.249299,-0.166874,-0.105671,0.061203,0.461882,-0.364008,-13.341988,-11.774469,208.8,227.4,2.071529,2.238403,0.7949,0.9640


In [57]:
df_mod[df_features.columns] = df_features

df_mod

,Season,Round,Result,Team A ID,Team A,Team B ID,Team B,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,EFG%,EFGD%,TOR,TORD,ORB,FTR,FTRD,ADJ T.,WAB,Past Year BARTHAG,Past 4 Years BARTHAG,Past Year ADJEM,Past 4 Years ADJEM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Team A ADJOE Team B ADJDE,Team B ADJOE Team A ADJDE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A BARTHAG,Team B BARTHAG
0,2012,1,1,1124,Baylor,1355,S Dakota St,0.0,1.250000,0.006629,2.7,-7.1,9.8,0.1420,-1.2,-2.7,5.0,2.1,7.1,-3.4,2.2,0.4,7.4,0.0703,0.428300,2.4,20.000,2.338760,0.067615,0.002898,-0.064718,0.583440,0.117964,7.605073,7.704642,215.1,205.3,2.102757,2.035141,0.9090,0.7670
1,2012,1,1,1160,Colorado,1424,UNLV,-1.0,-1.000000,-0.093750,-5.1,1.9,-7.0,-0.1138,-3.9,0.9,-0.3,-2.3,-3.3,7.2,-0.9,-4.0,-3.1,-0.0848,-0.168775,-4.6,-8.075,-1.031018,-0.086742,-0.061467,0.025275,-3.751519,-0.226425,-7.098771,-6.444547,196.5,203.5,1.942409,2.029151,0.7615,0.8753
2,2012,1,1,1211,Gonzaga,1452,West Virginia,0.0,-0.750000,0.212702,-0.3,-1.2,0.9,0.0132,4.7,-2.8,0.6,-0.7,-6.4,8.9,-4.7,0.7,2.3,-0.0286,-0.026625,-3.0,-2.850,0.274774,0.029706,0.004889,-0.024816,0.670748,0.164332,6.727094,5.540592,206.1,205.2,2.033816,2.004111,0.8612,0.8480
3,2012,1,1,1231,Indiana,1308,New Mexico St,0.0,0.000000,0.030303,13.0,-0.9,13.9,0.1917,4.0,-0.1,-1.6,-0.1,-5.2,-6.2,1.0,-3.3,7.6,0.1886,0.008000,7.6,1.450,1.508085,0.104455,0.097244,-0.007210,-3.018990,0.115812,7.908734,8.270663,216.5,202.6,2.116803,2.012348,0.9184,0.7267
4,2012,1,1,1235,Iowa St,1163,Connecticut,-7.0,-3.250000,0.081439,0.9,1.0,-0.1,-0.0023,2.7,3.9,-0.8,0.7,-4.6,4.3,0.8,3.0,1.4,-0.1929,-0.200225,-13.4,-13.550,0.155641,0.007995,0.021385,0.013390,2.322124,-0.035981,3.232739,3.208781,205.5,205.6,2.038216,2.030221,0.8553,0.8576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1505,2024,4,-1,1181,Duke,1301,NC State,1.0,2.000000,0.138889,6.8,-4.4,11.2,0.1313,4.5,-1.5,0.6,-1.0,3.2,1.3,-4.3,-1.1,4.0,0.0800,0.136625,6.0,10.200,0.102572,0.121062,0.073952,-0.047109,-0.912177,0.151976,5.024188,4.103541,222.1,210.9,2.206684,2.085623,0.9262,0.7949
1506,2024,4,-1,1397,Tennessee,1345,Purdue,2.0,0.333333,-0.128788,-10.6,-3.5,-7.1,-0.0265,-4.5,-2.3,-1.9,4.9,-5.0,-8.5,12.8,1.6,-5.2,0.0068,-0.016675,-0.5,-1.725,-0.279607,-0.035856,-0.077972,-0.042117,1.337846,-0.133075,-5.394708,-4.408191,210.3,217.4,2.099227,2.135083,0.9375,0.9640
1507,2024,5,-1,1104,Alabama,1163,Connecticut,-4.0,-0.666667,-0.255515,-1.7,8.2,-9.9,-0.0577,-0.8,4.8,1.1,-0.6,-1.6,1.9,7.1,8.0,-7.5,-0.0095,-0.001675,-3.2,-0.525,-0.392997,-0.059663,0.001401,0.061064,7.406847,-0.265038,-11.210705,-10.392373,219.0,228.9,2.178061,2.237724,0.9115,0.9692
1508,2024,5,-1,1301,NC State,1345,Purdue,0.0,-1.333333,-0.267677,-12.1,6.5,-18.6,-0.1691,-5.3,2.8,-2.8,3.9,-8.8,-10.3,8.9,0.3,-11.7,-0.1061,-0.129800,-9.8,-9.200,-1.249299,-0.166874,-0.105671,0.061203,0.461882,-0.364008,-13.341988,-11.774469,208.8,227.4,2.071529,2.238403,0.7949,0.9640


In [58]:
df_mod.to_parquet('../data/preprocessed/model_data/model_data.parquet')

'Done'

'Done'